# LectureLens — Cloud Run on Google Colab

Run **LectureLens** entirely on Google's cloud servers so **zero YouTube video files are downloaded or stored on your laptop**.
All temporary videos, frames, and caches live in Colab's cloud disk (`/content/lecturelens_cache`), and only the finished **PDF / Word (`.docx`)** notes download to your device (or save straight to Google Drive).

### Quick Steps
1. Run **Step 1** to install LectureLens & FFmpeg inside this Colab cloud VM.
2. Paste your YouTube URL in **Step 2**, pick your similarity preset (e.g. `strict_overlap` for `<=30%` overlap), and run the cell.
3. *(Optional)* Run **Step 3** to open the full interactive **LectureLens Web Studio** directly from Colab.

In [ ]:
#@title Step 1: Install LectureLens in Google Colab (Routes Cache to Cloud Storage)
GITHUB_REPO_URL = "https://github.com/manthansonarkar/LectureLens.git"  #@param {type:"string"}
#@markdown Clones directly from GitHub onto Colab's cloud VM (or clear the field to upload a local `.zip`).

import os
import shutil
import subprocess
from pathlib import Path

# Route all temporary videos & job caches strictly inside Colab's cloud VM
os.environ["LECTURELENS_CACHE_DIR"] = "/content/lecturelens_cache"
Path("/content/lecturelens_cache").mkdir(parents=True, exist_ok=True)

repo_dir = Path("/content/LectureLens")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

if GITHUB_REPO_URL.strip():
    subprocess.run(["git", "clone", GITHUB_REPO_URL.strip(), str(repo_dir)], check=True)
else:
    from google.colab import files
    import zipfile
    print("Upload LectureLens.zip:")
    uploaded = files.upload()
    zip_name = next(iter(uploaded.keys()))
    extract_root = Path("/content/_ll_extract")
    if extract_root.exists():
        shutil.rmtree(extract_root)
    with zipfile.ZipFile(zip_name, "r") as zf:
        zf.extractall(extract_root)
    candidates = list(extract_root.rglob("pyproject.toml"))
    assert candidates, "Could not find pyproject.toml inside the uploaded zip!"
    shutil.move(str(candidates[0].parent), str(repo_dir))

os.chdir(repo_dir)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "nodejs"], check=True)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "--no-deps", "-e", "."], check=True)
print("Ready! Cache routed to /content/lecturelens_cache (Colab cloud storage).")

In [ ]:
#@title Step 2: Paste YouTube Link & Export PDF / DOCX
YOUTUBE_URL = "https://www.youtube.com/watch?v=-1YVWlYPR1c"  #@param {type:"string"}
PRESET = "strict_overlap"  #@param ["smart", "strict_overlap", "detailed"]
MAX_SIMILARITY = 0.30  #@param {type:"slider", min:0.15, max:0.90, step:0.01}
SCROLL_OVERLAP = 0.28  #@param {type:"slider", min:0.10, max:0.75, step:0.01}
PDF_LAYOUT = "study_full"  #@param ["study_full", "cornell_notes", "handout_2x1", "grid_2x2"]
VIDEO_QUALITY = "1080p"  #@param ["720p", "1080p", "best"]
ENABLE_OCR = True  #@param {type:"boolean"}
AUTO_CLEANUP_VIDEO = True  #@param {type:"boolean"}
DOWNLOAD_PDF = True  #@param {type:"boolean"}
DOWNLOAD_DOCX = True  #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE = False  #@param {type:"boolean"}

import os
import shutil
import subprocess
from pathlib import Path
from google.colab import files

os.environ["LECTURELENS_CACHE_DIR"] = "/content/lecturelens_cache"

out_dir = Path("/content/lecturelens_outputs")
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    "lecturelens", "extract", YOUTUBE_URL.strip(),
    "--out", str(out_dir),
    "--preset", PRESET,
    "--max-similarity", str(MAX_SIMILARITY),
    "--scroll-overlap", str(SCROLL_OVERLAP),
    "--pdf-layout", PDF_LAYOUT,
    "--quality", VIDEO_QUALITY,
]
if not ENABLE_OCR:
    cmd.append("--no-ocr")
if AUTO_CLEANUP_VIDEO:
    cmd.append("--cleanup")
if DOWNLOAD_PDF:
    cmd.extend(["--format", "pdf"])
if DOWNLOAD_DOCX:
    cmd.extend(["--format", "docx"])

subprocess.run(cmd, check=True)

drive_dir = None
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive/LectureLens_Notes")
    drive_dir.mkdir(parents=True, exist_ok=True)

for doc_file in sorted(out_dir.glob("*_notes.*")):
    if drive_dir is not None:
        dest = drive_dir / doc_file.name
        shutil.copy2(doc_file, dest)
        print("Saved to Google Drive:", dest)
    else:
        print("Downloading final document:", doc_file.name)
        files.download(str(doc_file))

In [ ]:
#@title Step 3 (Optional): Launch Full LectureLens Web Studio in Google Colab
import os
import subprocess
import time
from google.colab import output

os.environ["LECTURELENS_CACHE_DIR"] = "/content/lecturelens_cache"
proc = subprocess.Popen(["lecturelens", "serve", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(3)
studio_url = output.eval_js("google.colab.kernel.proxyPort(8000)")
print("Open your cloud LectureLens Studio here (runs on Google Colab, 0 laptop storage):")
print(studio_url)
output.serve_kernel_port_as_window(8000)